In [ ]:
!pip install -q sentence-transformers faiss-cpu pyarrow transformers accelerate peft bitsandbytes joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 118.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.4 MB/s eta 0:00:00


In [ ]:
import os
import re
import ast
import gc
import json
import time
import random
import warnings
import unicodedata
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

import faiss
import joblib

import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 250)
pd.set_option("display.max_colwidth", 400)
pd.set_option("display.width", 240)

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA L4


In [ ]:
LOCAL_ARTIFACTS_DIR = Path("artifacts")
DRIVE_ARTIFACTS_DIR = Path("/content/drive/MyDrive/tablewise/artifacts_new")

if DRIVE_ARTIFACTS_DIR.exists():
    ARTIFACTS_DIR = DRIVE_ARTIFACTS_DIR
elif LOCAL_ARTIFACTS_DIR.exists():
    ARTIFACTS_DIR = LOCAL_ARTIFACTS_DIR
else:
    raise FileNotFoundError("Nu am găsit folderul artifacts. Verifică path-ul.")

FAISS_DIR = ARTIFACTS_DIR / "faiss"

MAPPING_PATH = FAISS_DIR / "restaurant_index_mapping.parquet"
EMBEDDINGS_PATH = FAISS_DIR / "restaurant_embeddings.npy"
FAISS_INDEX_PATH = FAISS_DIR / "restaurant_faiss.index"
FAISS_METADATA_PATH = FAISS_DIR / "metadata.json"

SLM_ADAPTER_DIR = ARTIFACTS_DIR / "slm_query_parser" / "qwen2_5_1_5b_query_parser_lora"

REWARD_MODEL_PATH = ARTIFACTS_DIR / "feedback_rlhf" / "reward_model" / "feedback_reward_model.joblib"
REWARD_FEATURES_PATH = ARTIFACTS_DIR / "feedback_rlhf" / "reward_model" / "reward_feature_columns.json"

OUTPUT_DIR = ARTIFACTS_DIR / "final_pipeline"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("ARTIFACTS_DIR:", ARTIFACTS_DIR)
print("FAISS_DIR:", FAISS_DIR)
print("SLM_ADAPTER_DIR:", SLM_ADAPTER_DIR)
print("REWARD_MODEL_PATH:", REWARD_MODEL_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)

ARTIFACTS_DIR: /content/drive/MyDrive/tablewise/artifacts_new
FAISS_DIR: /content/drive/MyDrive/tablewise/artifacts_new/faiss
SLM_ADAPTER_DIR: /content/drive/MyDrive/tablewise/artifacts_new/slm_query_parser/qwen2_5_1_5b_query_parser_lora
REWARD_MODEL_PATH: /content/drive/MyDrive/tablewise/artifacts_new/feedback_rlhf/reward_model/feedback_reward_model.joblib
OUTPUT_DIR: /content/drive/MyDrive/tablewise/artifacts_new/final_pipeline


In [ ]:
assert MAPPING_PATH.exists(), f"Missing mapping: {MAPPING_PATH}"
assert EMBEDDINGS_PATH.exists(), f"Missing embeddings: {EMBEDDINGS_PATH}"
assert FAISS_INDEX_PATH.exists(), f"Missing FAISS index: {FAISS_INDEX_PATH}"

mapping_df = pd.read_parquet(MAPPING_PATH)
embeddings = np.load(EMBEDDINGS_PATH, mmap_mode="r")
faiss_index = faiss.read_index(str(FAISS_INDEX_PATH))

if FAISS_METADATA_PATH.exists():
    with open(FAISS_METADATA_PATH, "r", encoding="utf-8") as f:
        faiss_metadata = json.load(f)
else:
    faiss_metadata = {}

print("mapping_df:", mapping_df.shape)
print("embeddings:", embeddings.shape)
print("FAISS ntotal:", faiss_index.ntotal)

assert len(mapping_df) == embeddings.shape[0]
assert len(mapping_df) == faiss_index.ntotal

display(mapping_df.head(3))

mapping_df: (1033798, 31)
embeddings: (1033798, 384)
FAISS ntotal: 1033798


,faiss_idx,restaurant_id,name,country,region,province,city,city_filled,city_filled_norm,city_source,address,latitude,longitude,rating,price_level,price_bucket,top_tags_text,top_tags_norm_list,meals_text,meals_norm_list,features_text,features_norm_list,special_diets_text,special_diets_norm_list,popularity_detailed,popularity_rank_num,popularity_total_num,popularity_score,profile_quality_score,short_profile,profile_text
0,0,g10001637-d10002227,Le 147,France,Nouvelle-Aquitaine,Haute-Vienne,Saint-Jouvent,Saint-Jouvent,saint-jouvent,original_city,"10 Maison Neuve, 87510 Saint-Jouvent France",45.961674,1.169131,4.0,€,cheap,"Cheap Eats, French","[cheap eats, french]","Lunch, Dinner","[lunch, dinner]","Reservations, Seating, Wheelchair Accessible, Serves Alcohol, Accepts Credit Cards, Table Service","[reservations, seating, wheelchair accessible, serves alcohol, accepts credit cards, table service]",Unknown,[],#1 of 2 Restaurants in Saint-Jouvent,1.0,2.0,0.36907,11,"Le 147 | Saint-Jouvent, France | rating=4.0 | price=cheap | tags=Cheap Eats, French","Le 147 is a restaurant located in Saint-Jouvent, Haute-Vienne, Nouvelle-Aquitaine, France. Address: 10 Maison Neuve, 87510 Saint-Jouvent France. Average rating: 4.0 out of 5. Price category: cheap. Original price level: €. Cuisines and tags: Cheap Eats, French. Meals served: Lunch, Dinner. Features: Reservations, Seating, Wheelchair Accessible, Serves Alcohol, Accepts Credit Cards, Table Servi..."
1,1,g10001637-d14975787,Le Saint Jouvent,France,Nouvelle-Aquitaine,Haute-Vienne,Saint-Jouvent,Saint-Jouvent,saint-jouvent,original_city,"16 Place de l Eglise, 87510 Saint-Jouvent France",45.957040,1.205480,4.0,€,cheap,Cheap Eats,[cheap eats],Unknown,[],Unknown,[],Unknown,[],#2 of 2 Restaurants in Saint-Jouvent,2.0,2.0,0.00000,9,"Le Saint Jouvent | Saint-Jouvent, France | rating=4.0 | price=cheap | tags=Cheap Eats","Le Saint Jouvent is a restaurant located in Saint-Jouvent, Haute-Vienne, Nouvelle-Aquitaine, France. Address: 16 Place de l Eglise, 87510 Saint-Jouvent France. Average rating: 4.0 out of 5. Price category: cheap. Original price level: €. Cuisines and tags: Cheap Eats. Popularity: #2 of 2 Restaurants in Saint-Jouvent."
2,2,g10002858-d4586832,Au Bout du Pont,France,Centre-Val de Loire,Berry,Rivarennes,Rivarennes,rivarennes,original_city,"2 rue des Dames, 36800 Rivarennes France",46.635895,1.386133,5.0,€,cheap,"Cheap Eats, French, European","[cheap eats, french, european]","Dinner, Lunch, Drinks","[dinner, lunch, drinks]","Reservations, Seating, Table Service, Wheelchair Accessible","[reservations, seating, table service, wheelchair accessible]",Unknown,[],#1 of 1 Restaurant in Rivarennes,1.0,1.0,NaN,11,"Au Bout du Pont | Rivarennes, France | rating=5.0 | price=cheap | tags=Cheap Eats, French, European","Au Bout du Pont is a restaurant located in Rivarennes, Berry, Centre-Val de Loire, France. Address: 2 rue des Dames, 36800 Rivarennes France. Average rating: 5.0 out of 5. Price category: cheap. Original price level: €. Cuisines and tags: Cheap Eats, French, European. Meals served: Dinner, Lunch, Drinks. Features: Reservations, Seating, Table Service, Wheelchair Accessible. Popularity: #1 of 1..."


In [ ]:
EMBEDDING_MODEL_NAME = faiss_metadata.get(
    "embedding_model",
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Loading embedding model:", EMBEDDING_MODEL_NAME)

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("Embedding model loaded.")

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.


In [ ]:
assert REWARD_MODEL_PATH.exists(), f"Missing reward model: {REWARD_MODEL_PATH}"

reward_model = joblib.load(REWARD_MODEL_PATH)

if REWARD_FEATURES_PATH.exists():
    with open(REWARD_FEATURES_PATH, "r", encoding="utf-8") as f:
        reward_config = json.load(f)
else:
    reward_config = {}

FEATURE_COLS = reward_config.get(
    "feature_cols",
    [
        "semantic_score_norm",
        "metadata_score_norm",
        "rating_score",
        "popularity_score_norm",
        "soft_constraint_score",
        "hard_constraint_score",
    ],
)

FEEDBACK_RERANK_WEIGHTS = reward_config.get(
    "feedback_rerank_weights",
    {
        "original_rerank": 0.70,
        "reward_score": 0.30,
    },
)

print("Reward model loaded.")
print("FEATURE_COLS:", FEATURE_COLS)
print("FEEDBACK_RERANK_WEIGHTS:", FEEDBACK_RERANK_WEIGHTS)

Reward model loaded.
FEATURE_COLS: ['semantic_score_norm', 'metadata_score_norm', 'rating_score', 'popularity_score_norm', 'soft_constraint_score', 'hard_constraint_score']
FEEDBACK_RERANK_WEIGHTS: {'original_rerank': 0.7, 'reward_score': 0.3}


In [ ]:
assert REWARD_MODEL_PATH.exists(), f"Missing reward model: {REWARD_MODEL_PATH}"

reward_model = joblib.load(REWARD_MODEL_PATH)

if REWARD_FEATURES_PATH.exists():
    with open(REWARD_FEATURES_PATH, "r", encoding="utf-8") as f:
        reward_config = json.load(f)
else:
    reward_config = {}

FEATURE_COLS = reward_config.get(
    "feature_cols",
    [
        "semantic_score_norm",
        "metadata_score_norm",
        "rating_score",
        "popularity_score_norm",
        "soft_constraint_score",
        "hard_constraint_score",
    ],
)

FEEDBACK_RERANK_WEIGHTS = reward_config.get(
    "feedback_rerank_weights",
    {
        "original_rerank": 0.70,
        "reward_score": 0.30,
    },
)

print("Reward model loaded.")
print("FEATURE_COLS:", FEATURE_COLS)
print("FEEDBACK_RERANK_WEIGHTS:", FEEDBACK_RERANK_WEIGHTS)

Reward model loaded.
FEATURE_COLS: ['semantic_score_norm', 'metadata_score_norm', 'rating_score', 'popularity_score_norm', 'soft_constraint_score', 'hard_constraint_score']
FEEDBACK_RERANK_WEIGHTS: {'original_rerank': 0.7, 'reward_score': 0.3}


In [ ]:
def strip_accents(text):
    if text is None or pd.isna(text):
        return ""

    text = str(text)
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))

    return text


def normalize_text(x):
    if x is None or pd.isna(x):
        return ""

    s = strip_accents(str(x)).lower().strip()
    s = re.sub(r"[^a-z0-9\s\-]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    return s


def safe_value(x, default="Unknown"):
    if x is None or pd.isna(x):
        return default

    s = str(x).strip()

    if not s or s.lower() in {"nan", "none", "null"}:
        return default

    return s


def contains_phrase(text_norm, phrase_norm):
    if not text_norm or not phrase_norm:
        return False

    pattern = rf"(?<![a-z0-9]){re.escape(phrase_norm)}(?![a-z0-9])"
    return re.search(pattern, text_norm) is not None


def parse_possible_list(x):
    if x is None:
        return []

    if isinstance(x, float) and pd.isna(x):
        return []

    if isinstance(x, np.ndarray):
        values = x.tolist()
    elif isinstance(x, (list, tuple, set)):
        values = list(x)
    else:
        s = str(x).strip()

        if not s or s.lower() in {"nan", "none", "null", "unknown"}:
            return []

        if s.startswith("[") and s.endswith("]"):
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, (list, tuple, set)):
                    values = list(parsed)
                else:
                    values = [s]
            except Exception:
                values = re.split(r"[,|;/]", s)
        else:
            values = re.split(r"[,|;/]", s)

    cleaned = []
    seen = set()

    for item in values:
        item_s = safe_value(item, default="").strip()
        item_norm = normalize_text(item_s)

        if item_norm and item_norm not in seen:
            cleaned.append(item_s)
            seen.add(item_norm)

    return cleaned

In [ ]:
df = mapping_df.copy()

list_base_cols = [
    "top_tags",
    "meals",
    "features",
    "special_diets",
]

for base in list_base_cols:
    text_col = f"{base}_text"
    list_col = f"{base}_list"
    norm_list_col = f"{base}_norm_list"

    if list_col not in df.columns:
        if text_col in df.columns:
            df[list_col] = df[text_col].apply(parse_possible_list)
        else:
            df[list_col] = [[] for _ in range(len(df))]
    else:
        df[list_col] = df[list_col].apply(parse_possible_list)

    df[norm_list_col] = df[list_col].apply(
        lambda xs: [normalize_text(x) for x in xs if normalize_text(x)]
    )

for col in ["country", "region", "province", "city", "city_filled"]:
    if col in df.columns:
        df[f"{col}_norm"] = df[col].apply(normalize_text)

df["city_filled_norm"] = df["city_filled"].apply(normalize_text)
df["country_norm"] = df["country"].apply(normalize_text)

df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
df["rating_score"] = (df["rating"] / 5.0).clip(0, 1).fillna(0.5)

df["popularity_score"] = pd.to_numeric(df["popularity_score"], errors="coerce").clip(0, 1).fillna(0.5)

if "price_bucket" in df.columns:
    df["price_bucket"] = df["price_bucket"].apply(lambda x: normalize_text(x) if pd.notna(x) else None)

print("Prepared df:", df.shape)
display(df[["name", "country", "city_filled", "price_bucket", "rating", "rating_score", "popularity_score"]].head())

Prepared df: (1033798, 40)


,name,country,city_filled,price_bucket,rating,rating_score,popularity_score
0,Le 147,France,Saint-Jouvent,cheap,4.0,0.8,0.36907
1,Le Saint Jouvent,France,Saint-Jouvent,cheap,4.0,0.8,0.00000
2,Au Bout du Pont,France,Rivarennes,cheap,5.0,1.0,0.50000
3,Le Relais de Naiade,France,Lacelle,cheap,4.0,0.8,0.50000
4,Relais Du MontSeigne,France,Saint-Laurent-de-Levezou,mid,4.5,0.9,0.50000


In [ ]:
SLM_BASE_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

assert SLM_ADAPTER_DIR.exists(), f"Missing SLM adapter: {SLM_ADAPTER_DIR}"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    bnb_4bit_use_double_quant=True,
)

slm_tokenizer = AutoTokenizer.from_pretrained(
    SLM_ADAPTER_DIR,
    trust_remote_code=True,
)

if slm_tokenizer.pad_token is None:
    slm_tokenizer.pad_token = slm_tokenizer.eos_token

slm_base_model = AutoModelForCausalLM.from_pretrained(
    SLM_BASE_MODEL_NAME,
    quantization_config=bnb_config if torch.cuda.is_available() else None,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)

slm_model = PeftModel.from_pretrained(
    slm_base_model,
    SLM_ADAPTER_DIR,
)

slm_model.eval()

print("Fine-tuned SLM parser loaded.")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Fine-tuned SLM parser loaded.


In [ ]:
SYSTEM_PROMPT = """You are a query parser for a restaurant recommendation system.
Extract structured restaurant search constraints from the user query.
Return valid JSON only. Do not explain anything."""

USER_PROMPT_TEMPLATE = """Return valid JSON only with this schema:
{{
  "city": string or null,
  "country": string or null,
  "price_bucket": "cheap" or "mid" or "expensive" or null,
  "tags": list of strings,
  "dietary": list of strings,
  "matched_meals": list of strings,
  "matched_features": list of strings
}}

Allowed values:
price_bucket: cheap, mid, expensive
tags: italian, french, spanish, greek, portuguese, german, seafood, steakhouse, asian, indian, mexican, mediterranean, fast food, coffee, bar, fine dining, tapas
dietary: vegetarian friendly, vegan options, gluten free options
matched_meals: breakfast, brunch, lunch, dinner, drinks
matched_features: outdoor seating, reservations, delivery, wheelchair accessible, family friendly, romantic

User query:
{query}

JSON:"""


ALLOWED_PRICE_BUCKETS = ["cheap", "mid", "expensive"]

ALLOWED_TAGS = [
    "italian", "french", "spanish", "greek", "portuguese", "german",
    "seafood", "steakhouse", "asian", "indian", "mexican",
    "mediterranean", "fast food", "coffee", "bar", "fine dining", "tapas"
]

ALLOWED_DIETARY = [
    "vegetarian friendly", "vegan options", "gluten free options"
]

ALLOWED_MEALS = [
    "breakfast", "brunch", "lunch", "dinner", "drinks"
]

ALLOWED_FEATURES = [
    "outdoor seating", "reservations", "delivery",
    "wheelchair accessible", "family friendly", "romantic"
]


def make_empty_parsed_query():
    return {
        "city": None,
        "country": None,
        "price_bucket": None,
        "tags": [],
        "dietary": [],
        "matched_meals": [],
        "matched_features": [],
    }


def build_slm_messages(query):
    return [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": USER_PROMPT_TEMPLATE.format(query=query),
        },
    ]


def build_slm_prompt(query):
    messages = build_slm_messages(query)

    if hasattr(slm_tokenizer, "apply_chat_template"):
        return slm_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    return (
        f"SYSTEM: {messages[0]['content']}\n\n"
        f"USER: {messages[1]['content']}\n\n"
        f"ASSISTANT:"
    )


def extract_json_from_text(text):
    if text is None:
        return None

    s = str(text).strip()

    try:
        return json.loads(s)
    except Exception:
        pass

    m = re.search(r"\{.*\}", s, flags=re.DOTALL)

    if m:
        try:
            return json.loads(m.group(0))
        except Exception:
            return None

    return None


def validate_and_repair_parsed_query(obj):
    if not isinstance(obj, dict):
        return None

    repaired = make_empty_parsed_query()

    city = obj.get("city")
    country = obj.get("country")
    price = obj.get("price_bucket")

    city_norm = normalize_text(city) if city else None
    country_norm = normalize_text(country) if country else None

    repaired["city"] = city_norm if city_norm else None
    repaired["country"] = country_norm if country_norm else None
    repaired["price_bucket"] = price if price in ALLOWED_PRICE_BUCKETS else None

    for key, allowed in [
        ("tags", ALLOWED_TAGS),
        ("dietary", ALLOWED_DIETARY),
        ("matched_meals", ALLOWED_MEALS),
        ("matched_features", ALLOWED_FEATURES),
    ]:
        values = obj.get(key, [])

        if values is None:
            values = []

        if isinstance(values, str):
            values = [values]

        if not isinstance(values, list):
            values = []

        cleaned_values = []

        for v in values:
            v_norm = normalize_text(v)
            if v_norm in allowed:
                cleaned_values.append(v_norm)

        repaired[key] = sorted(set(cleaned_values))

    return repaired


def slm_parse_query(query, max_new_tokens=180):
    prompt_text = build_slm_prompt(query)

    inputs = slm_tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    )

    device = next(slm_model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        generated = slm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=slm_tokenizer.eos_token_id,
            eos_token_id=slm_tokenizer.eos_token_id,
        )

    input_len = inputs["input_ids"].shape[-1]
    new_tokens = generated[0][input_len:]

    raw_output = slm_tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    ).strip()

    parsed_obj = extract_json_from_text(raw_output)
    validated = validate_and_repair_parsed_query(parsed_obj)

    return {
        "query": query,
        "raw_output": raw_output,
        "parsed_json": parsed_obj,
        "validated_json": validated,
        "valid_json": validated is not None,
    }

In [ ]:
test_queries_parser = [
    "Can you recommend me a cheap vegetarian Italian restaurant in Rome?",
    "I want an expensive fine dining place in Milan.",
    "Find me a vegetarian brunch place in Barcelona.",
    "Can you suggest tapas in Madrid?",
    "I need coffee and breakfast in Athens.",
    "Somewhere romantic for dinner in Paris, not too expensive.",
]

for q in test_queries_parser:
    out = slm_parse_query(q)
    print("=" * 120)
    print("QUERY:", q)
    print("RAW:", out["raw_output"])
    print("VALIDATED:")
    print(json.dumps(out["validated_json"], indent=2, ensure_ascii=False))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


QUERY: Can you recommend me a cheap vegetarian Italian restaurant in Rome?
RAW: {"city": "rome", "country": null, "price_bucket": "cheap", "tags": ["italian"], "dietary": ["vegetarian friendly"], "matched_meals": [], "matched_features": []}
VALIDATED:
{
  "city": "rome",
  "country": null,
  "price_bucket": "cheap",
  "tags": [
    "italian"
  ],
  "dietary": [
    "vegetarian friendly"
  ],
  "matched_meals": [],
  "matched_features": []
}
QUERY: I want an expensive fine dining place in Milan.
RAW: {"city": "milan", "country": null, "price_bucket": "expensive", "tags": ["fine dining"], "dietary": [], "matched_meals": [], "matched_features": []}
VALIDATED:
{
  "city": "milan",
  "country": null,
  "price_bucket": "expensive",
  "tags": [
    "fine dining"
  ],
  "dietary": [],
  "matched_meals": [],
  "matched_features": []
}
QUERY: Find me a vegetarian brunch place in Barcelona.
RAW: {"city": "barcelona", "country": null, "price_bucket": null, "tags": [], "dietary": ["vegetarian frien

In [ ]:
def list_contains_term(row_list, term):
    if not isinstance(row_list, list):
        return False

    term_norm = normalize_text(term)

    if not term_norm:
        return False

    for item in row_list:
        item_norm = normalize_text(item)

        if not item_norm:
            continue

        if term_norm == item_norm:
            return True

        if term_norm in item_norm:
            return True

        if item_norm in term_norm and len(item_norm) >= 4:
            return True

    return False


def compute_metadata_score_for_row(row, parsed_query):
    score = 0.0
    reasons = []

    price_bucket = parsed_query.get("price_bucket")
    tags = parsed_query.get("tags", [])
    dietary = parsed_query.get("dietary", [])
    meals = parsed_query.get("matched_meals", [])
    features = parsed_query.get("matched_features", [])

    if price_bucket and row.get("price_bucket") == price_bucket:
        score += 2.0
        reasons.append(f"price={price_bucket}")

    row_tags = row.get("top_tags_norm_list", [])
    row_diets = row.get("special_diets_norm_list", [])
    row_meals = row.get("meals_norm_list", [])
    row_features = row.get("features_norm_list", [])

    for tag in tags:
        if list_contains_term(row_tags, tag):
            score += 2.0
            reasons.append(f"tag={tag}")

    for diet in dietary:
        if (
            list_contains_term(row_diets, diet)
            or list_contains_term(row_tags, diet)
            or list_contains_term(row_features, diet)
        ):
            score += 2.0
            reasons.append(f"diet={diet}")

    for meal in meals:
        if list_contains_term(row_meals, meal) or list_contains_term(row_tags, meal):
            score += 1.0
            reasons.append(f"meal={meal}")

    for feature in features:
        if list_contains_term(row_features, feature) or list_contains_term(row_tags, feature):
            score += 1.0
            reasons.append(f"feature={feature}")

    return score, sorted(set(reasons))


def add_metadata_scores(df_in, parsed_query):
    current = df_in.copy()

    if len(current) == 0:
        current["metadata_score"] = pd.Series(dtype=float)
        current["metadata_match_reasons"] = pd.Series(dtype=object)
        return current

    scores_and_reasons = current.apply(
        lambda row: compute_metadata_score_for_row(row, parsed_query),
        axis=1,
    )

    current["metadata_score"] = scores_and_reasons.apply(lambda x: x[0])
    current["metadata_match_reasons"] = scores_and_reasons.apply(lambda x: x[1])

    return current

In [ ]:
def filter_candidates_from_parsed_query(parsed_query, df_in, min_metadata_results=50, verbose=True):
    current = df_in.copy()

    if verbose:
        print("Initial rows:", len(current))
        print("Parsed query:")
        print(json.dumps(parsed_query, indent=2, ensure_ascii=False))

    city = parsed_query.get("city")
    country = parsed_query.get("country")

    if city:
        before = len(current)
        current = current[current["city_filled_norm"] == city].copy()

        if verbose:
            print(f"City filter city={city}: {before} -> {len(current)}")

        if len(current) == 0:
            current = add_metadata_scores(current, parsed_query)
            return current.reset_index(drop=True)

    if country:
        before = len(current)
        current = current[current["country_norm"] == country].copy()

        if verbose:
            print(f"Country filter country={country}: {before} -> {len(current)}")

        if len(current) == 0:
            current = add_metadata_scores(current, parsed_query)
            return current.reset_index(drop=True)

    current = add_metadata_scores(current, parsed_query)

    has_metadata_constraints = bool(
        parsed_query.get("price_bucket")
        or parsed_query.get("tags")
        or parsed_query.get("dietary")
        or parsed_query.get("matched_meals")
        or parsed_query.get("matched_features")
    )

    if has_metadata_constraints:
        before = len(current)
        filtered = current[current["metadata_score"] > 0].copy()

        if verbose:
            print(f"Metadata score > 0 filter: {before} -> {len(filtered)}")

        if len(filtered) >= min_metadata_results:
            current = filtered

    if verbose:
        print("After filtering:", len(current))

    return current.reset_index(drop=True)

In [ ]:
def encode_query(query):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")

    return query_embedding[0]


def semantic_search_within_candidates(
    query,
    candidates_df,
    embeddings_array,
    top_k=100,
    score_chunk_size=200_000,
    verbose=True,
):
    if len(candidates_df) == 0:
        result = candidates_df.copy()
        result["semantic_score"] = pd.Series(dtype=float)
        return result

    query_embedding = encode_query(query)

    faiss_indices = candidates_df["faiss_idx"].to_numpy(dtype=np.int64)

    n = len(faiss_indices)
    scores = np.empty(n, dtype=np.float32)

    if verbose:
        print(f"Computing semantic scores for {n:,} candidates...")

    for start in range(0, n, score_chunk_size):
        end = min(start + score_chunk_size, n)

        idx_chunk = faiss_indices[start:end]
        emb_chunk = embeddings_array[idx_chunk]

        scores[start:end] = emb_chunk @ query_embedding

    k = min(top_k, n)

    if k == n:
        top_positions = np.argsort(-scores)
    else:
        top_positions_unsorted = np.argpartition(-scores, k - 1)[:k]
        top_positions = top_positions_unsorted[np.argsort(-scores[top_positions_unsorted])]

    result = candidates_df.iloc[top_positions].copy().reset_index(drop=True)
    result["semantic_score"] = scores[top_positions]

    return result

In [ ]:
RERANK_WEIGHTS = {
    "semantic": 0.40,
    "metadata": 0.20,
    "rating": 0.15,
    "popularity": 0.10,
    "soft_constraints": 0.15,
}


def normalize_semantic_score(series):
    s = pd.to_numeric(series, errors="coerce").fillna(0.0)
    return ((s + 1.0) / 2.0).clip(0, 1)


def normalize_metadata_score(series):
    s = pd.to_numeric(series, errors="coerce").fillna(0.0)
    max_val = s.max()

    if pd.isna(max_val) or max_val <= 0:
        return pd.Series(np.zeros(len(s)), index=s.index)

    return (s / max_val).clip(0, 1)


def compute_constraint_scores_for_row(row, parsed_query):
    hard_score = 1.0
    soft_score = 0.0
    max_soft_score = 0.0
    reasons = []

    city = parsed_query.get("city")
    country = parsed_query.get("country")
    price_bucket = parsed_query.get("price_bucket")
    tags = parsed_query.get("tags", [])
    dietary = parsed_query.get("dietary", [])
    meals = parsed_query.get("matched_meals", [])
    features = parsed_query.get("matched_features", [])

    if city:
        if row.get("city_filled_norm") == city:
            reasons.append(f"city={city}")
        else:
            hard_score = 0.0

    if country:
        if row.get("country_norm") == country:
            reasons.append(f"country={country}")
        else:
            hard_score = 0.0

    if price_bucket:
        max_soft_score += 1.0
        if row.get("price_bucket") == price_bucket:
            soft_score += 1.0
            reasons.append(f"price={price_bucket}")

    row_tags = row.get("top_tags_norm_list", [])
    row_diets = row.get("special_diets_norm_list", [])
    row_meals = row.get("meals_norm_list", [])
    row_features = row.get("features_norm_list", [])

    for tag in tags:
        max_soft_score += 1.0
        if list_contains_term(row_tags, tag):
            soft_score += 1.0
            reasons.append(f"tag={tag}")

    for diet in dietary:
        max_soft_score += 1.0
        if (
            list_contains_term(row_diets, diet)
            or list_contains_term(row_tags, diet)
            or list_contains_term(row_features, diet)
        ):
            soft_score += 1.0
            reasons.append(f"diet={diet}")

    for meal in meals:
        max_soft_score += 1.0
        if list_contains_term(row_meals, meal) or list_contains_term(row_tags, meal):
            soft_score += 1.0
            reasons.append(f"meal={meal}")

    for feature in features:
        max_soft_score += 1.0
        if list_contains_term(row_features, feature) or list_contains_term(row_tags, feature):
            soft_score += 1.0
            reasons.append(f"feature={feature}")

    if max_soft_score > 0:
        soft_constraint_score = soft_score / max_soft_score
    else:
        soft_constraint_score = 0.5

    return hard_score, soft_constraint_score, sorted(set(reasons))


def rerank_candidates(candidates_df, parsed_query, top_k=20):
    results = candidates_df.copy()

    if len(results) == 0:
        return results

    if "metadata_score" not in results.columns:
        results["metadata_score"] = 0.0

    results["semantic_score_norm"] = normalize_semantic_score(results["semantic_score"])
    results["metadata_score_norm"] = normalize_metadata_score(results["metadata_score"])
    results["rating_score"] = (pd.to_numeric(results["rating"], errors="coerce") / 5.0).clip(0, 1).fillna(0.5)
    results["popularity_score_norm"] = pd.to_numeric(results["popularity_score"], errors="coerce").clip(0, 1).fillna(0.5)

    constraint_values = results.apply(
        lambda row: compute_constraint_scores_for_row(row, parsed_query),
        axis=1,
    )

    results["hard_constraint_score"] = constraint_values.apply(lambda x: x[0])
    results["soft_constraint_score"] = constraint_values.apply(lambda x: x[1])
    results["constraint_match_reasons"] = constraint_values.apply(lambda x: x[2])

    results["final_rerank_score"] = (
        RERANK_WEIGHTS["semantic"] * results["semantic_score_norm"]
        + RERANK_WEIGHTS["metadata"] * results["metadata_score_norm"]
        + RERANK_WEIGHTS["rating"] * results["rating_score"]
        + RERANK_WEIGHTS["popularity"] * results["popularity_score_norm"]
        + RERANK_WEIGHTS["soft_constraints"] * results["soft_constraint_score"]
    )

    results["final_rerank_score"] = results["final_rerank_score"] * results["hard_constraint_score"]

    results = results.sort_values(
        [
            "final_rerank_score",
            "soft_constraint_score",
            "metadata_score_norm",
            "semantic_score_norm",
            "rating_score",
        ],
        ascending=False,
    ).reset_index(drop=True)

    results.insert(0, "rerank_position", np.arange(1, len(results) + 1))

    if top_k is not None:
        results = results.head(top_k).copy().reset_index(drop=True)

    return results

In [ ]:
def apply_feedback_reward_reranking(
    reranked_df,
    reward_model,
    feature_cols=FEATURE_COLS,
    top_k=5,
):
    df_out = reranked_df.copy()

    if len(df_out) == 0:
        return df_out

    for col in feature_cols:
        if col not in df_out.columns:
            df_out[col] = 0.0

        df_out[col] = pd.to_numeric(
            df_out[col],
            errors="coerce",
        ).fillna(0.0)

    if "final_rerank_score" not in df_out.columns:
        df_out["final_rerank_score"] = 0.0

    df_out["final_rerank_score"] = pd.to_numeric(
        df_out["final_rerank_score"],
        errors="coerce",
    ).fillna(0.0)

    df_out["reward_score"] = reward_model.predict_proba(
        df_out[feature_cols].values
    )[:, 1]

    df_out["feedback_rerank_score"] = (
        FEEDBACK_RERANK_WEIGHTS["original_rerank"] * df_out["final_rerank_score"].astype(float)
        + FEEDBACK_RERANK_WEIGHTS["reward_score"] * df_out["reward_score"].astype(float)
    )

    df_out = df_out.sort_values(
        ["feedback_rerank_score", "reward_score", "final_rerank_score"],
        ascending=False,
    ).reset_index(drop=True)

    df_out.insert(0, "feedback_rank_position", np.arange(1, len(df_out) + 1))

    if top_k is not None:
        df_out = df_out.head(top_k).copy().reset_index(drop=True)

    return df_out

In [ ]:
def retrieve_rerank_and_feedback_rank(
    query,
    parsed_query,
    retrieval_top_k=100,
    baseline_top_k=20,
    final_top_k=5,
    verbose=True,
):
    start_time = time.time()

    candidates = filter_candidates_from_parsed_query(
        parsed_query=parsed_query,
        df_in=df,
        min_metadata_results=50,
        verbose=verbose,
    )

    if len(candidates) == 0:
        return {
            "query": query,
            "parsed_query": parsed_query,
            "candidates": candidates,
            "baseline_results": candidates,
            "feedback_results": candidates,
            "elapsed_seconds": time.time() - start_time,
        }

    semantic_results = semantic_search_within_candidates(
        query=query,
        candidates_df=candidates,
        embeddings_array=embeddings,
        top_k=retrieval_top_k,
        verbose=verbose,
    )

    baseline_results = rerank_candidates(
        candidates_df=semantic_results,
        parsed_query=parsed_query,
        top_k=baseline_top_k,
    )

    feedback_results = apply_feedback_reward_reranking(
        reranked_df=baseline_results,
        reward_model=reward_model,
        top_k=final_top_k,
    )

    return {
        "query": query,
        "parsed_query": parsed_query,
        "candidates": candidates,
        "baseline_results": baseline_results,
        "feedback_results": feedback_results,
        "elapsed_seconds": time.time() - start_time,
    }

In [ ]:
def format_rating(x):
    if pd.isna(x):
        return "unknown"
    try:
        return f"{float(x):.1f}/5"
    except Exception:
        return "unknown"


def format_price_bucket(x):
    x = safe_value(x, default="unknown").lower()
    if x in {"cheap", "mid", "expensive"}:
        return x
    return "unknown"


def row_to_rag_evidence(row, rank=None):
    name = safe_value(row.get("name"))
    city = safe_value(row.get("city_filled"))
    country = safe_value(row.get("country"))
    rating = format_rating(row.get("rating"))
    price = format_price_bucket(row.get("price_bucket"))

    tags = safe_value(row.get("top_tags_text"))
    diets = safe_value(row.get("special_diets_text"))
    meals = safe_value(row.get("meals_text"))
    features = safe_value(row.get("features_text"))
    reasons = row.get("constraint_match_reasons", [])

    if not isinstance(reasons, list):
        reasons = []

    evidence = {
        "rank": int(rank) if rank is not None else None,
        "name": name,
        "location": f"{city}, {country}",
        "city": city,
        "country": country,
        "rating": rating,
        "price_bucket": price,
        "cuisines_and_tags": tags,
        "special_diets": diets,
        "meals": meals,
        "features": features,
        "match_reasons": reasons,
        "final_rerank_score": float(row.get("final_rerank_score", 0.0) or 0.0),
        "reward_score": float(row.get("reward_score", 0.0) or 0.0),
        "feedback_rerank_score": float(row.get("feedback_rerank_score", 0.0) or 0.0),
    }

    return evidence


def build_rag_context(reranked_df, top_n=5):
    if len(reranked_df) == 0:
        return {
            "restaurants": [],
            "context_text": "",
            "allowed_restaurant_names": [],
        }

    top_df = reranked_df.head(top_n).copy()

    restaurants = []

    for i, (_, row) in enumerate(top_df.iterrows(), start=1):
        restaurants.append(row_to_rag_evidence(row, rank=i))

    context_lines = []

    for r in restaurants:
        context_lines.append(f"[{r['rank']}] {r['name']}")
        context_lines.append(f"Location: {r['location']}")
        context_lines.append(f"Rating: {r['rating']}")
        context_lines.append(f"Price bucket: {r['price_bucket']}")

        if r["cuisines_and_tags"] != "Unknown":
            context_lines.append(f"Cuisines/tags: {r['cuisines_and_tags']}")

        if r["special_diets"] != "Unknown":
            context_lines.append(f"Special diets: {r['special_diets']}")

        if r["meals"] != "Unknown":
            context_lines.append(f"Meals: {r['meals']}")

        if r["features"] != "Unknown":
            context_lines.append(f"Features: {r['features']}")

        if r["match_reasons"]:
            context_lines.append(f"Matched constraints: {', '.join(r['match_reasons'])}")

        context_lines.append(f"Initial rerank score: {r['final_rerank_score']:.4f}")
        context_lines.append(f"Feedback reward score: {r['reward_score']:.4f}")
        context_lines.append("")

    context_text = "\n".join(context_lines).strip()

    allowed_restaurant_names = [r["name"] for r in restaurants]

    return {
        "restaurants": restaurants,
        "context_text": context_text,
        "allowed_restaurant_names": allowed_restaurant_names,
    }

In [ ]:
def describe_match_from_evidence(evidence, parsed_query):
    reasons = []

    price = parsed_query.get("price_bucket")
    tags = parsed_query.get("tags", [])
    dietary = parsed_query.get("dietary", [])
    meals = parsed_query.get("matched_meals", [])
    features = parsed_query.get("matched_features", [])

    evidence_price = evidence.get("price_bucket", "unknown")
    evidence_tags = normalize_text(evidence.get("cuisines_and_tags", ""))
    evidence_diets = normalize_text(evidence.get("special_diets", ""))
    evidence_meals = normalize_text(evidence.get("meals", ""))
    evidence_features = normalize_text(evidence.get("features", ""))

    if price and evidence_price == price:
        reasons.append(f"it matches the requested budget ({price})")

    for tag in tags:
        if contains_phrase(evidence_tags, normalize_text(tag)):
            reasons.append(f"it has a relevant cuisine/tag: {tag}")

    for diet in dietary:
        diet_norm = normalize_text(diet)
        if (
            contains_phrase(evidence_diets, diet_norm)
            or contains_phrase(evidence_tags, diet_norm)
            or contains_phrase(evidence_features, diet_norm)
        ):
            reasons.append(f"it supports the requested dietary preference: {diet}")

    for meal in meals:
        meal_norm = normalize_text(meal)
        if contains_phrase(evidence_meals, meal_norm) or contains_phrase(evidence_tags, meal_norm):
            reasons.append(f"it is suitable for {meal}")

    for feature in features:
        feature_norm = normalize_text(feature)
        if contains_phrase(evidence_features, feature_norm) or contains_phrase(evidence_tags, feature_norm):
            reasons.append(f"it has a relevant feature: {feature}")

    if not reasons:
        if evidence.get("rating", "unknown") != "unknown":
            reasons.append(f"it has a rating of {evidence.get('rating')}")
        else:
            reasons.append("it is one of the most relevant restaurants retrieved from the dataset")

    return reasons[:3]


def generate_template_answer(query, parsed_query, rag_context, max_recommendations=3):
    restaurants = rag_context["restaurants"][:max_recommendations]

    if len(restaurants) == 0:
        city = parsed_query.get("city")
        country = parsed_query.get("country")

        if city:
            return (
                f"I could not find matching restaurants in the dataset for the city “{city}”. "
                "You can try a nearby city or a broader request."
            )

        if country:
            return (
                f"I could not find matching restaurants in the dataset for “{country}”. "
                "You can try relaxing some constraints."
            )

        return (
            "I could not find sufficiently relevant restaurants in the dataset for this request. "
            "Try specifying a city, cuisine, or budget."
        )

    intro = "Here are a few relevant options based on the restaurants retrieved from the dataset:\n"
    lines = [intro]

    for i, evidence in enumerate(restaurants, start=1):
        name = evidence["name"]
        location = evidence["location"]
        rating = evidence["rating"]
        price = evidence["price_bucket"]

        reasons = describe_match_from_evidence(evidence, parsed_query)

        line = (
            f"{i}. **{name}** — {location}. "
            f"Rating: {rating}; price: {price}. "
            f"Why it matches: " + "; ".join(reasons) + "."
        )

        lines.append(line)

    lines.append(
        "\nThis recommendation is based only on the retrieved restaurants and their available metadata."
    )

    return "\n".join(lines)

In [ ]:
USE_HF_LLM_GENERATOR = True

HF_GENERATION_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
HF_MAX_NEW_TOKENS = 350

if USE_HF_LLM_GENERATOR:
    hf_tokenizer = AutoTokenizer.from_pretrained(
        HF_GENERATION_MODEL_NAME,
        trust_remote_code=True,
    )

    if hf_tokenizer.pad_token is None:
        hf_tokenizer.pad_token = hf_tokenizer.eos_token

    hf_model = AutoModelForCausalLM.from_pretrained(
        HF_GENERATION_MODEL_NAME,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
        trust_remote_code=True,
    )

    hf_model.eval()

    print("HF generation model loaded.")
else:
    hf_tokenizer = None
    hf_model = None
    print("HF generation disabled. Template fallback will be used.")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

HF generation model loaded.


In [ ]:
def build_strict_hf_prompt(query, parsed_query, rag_context, max_recommendations=3):
    allowed_names = rag_context["allowed_restaurant_names"][:max_recommendations]
    context_text = rag_context["context_text"]

    if len(allowed_names) == 0:
        user_content = f"""
User query:
{query}

Parsed query constraints:
{json.dumps(parsed_query, ensure_ascii=False, indent=2)}

Retrieved restaurant evidence:
No restaurants were retrieved.

Instructions:
- Say that no matching restaurants were found in the dataset.
- Do not recommend any restaurant.
- Answer in English.
- Keep it short and natural.
""".strip()
    else:
        user_content = f"""
User query:
{query}

Parsed query constraints:
{json.dumps(parsed_query, ensure_ascii=False, indent=2)}

Allowed restaurant names:
{json.dumps(allowed_names, ensure_ascii=False, indent=2)}

Retrieved restaurant evidence:
{context_text}

Strict grounding rules:
- Use ONLY the retrieved restaurant evidence.
- Mention ONLY restaurant names from the allowed restaurant names list.
- Recommend at most {max_recommendations} restaurants.
- Do NOT invent restaurant names.
- Do NOT invent unsupported facts.
- Do NOT infer vegan, gluten-free, romantic, family-friendly, outdoor seating, or other attributes unless they appear explicitly in the evidence.
- If a field is unknown, do not mention it.

Writing style:
- Answer in English.
- Sound like a helpful restaurant assistant.
- Keep the answer concise.
- Put every restaurant name in bold.
- Do not use markdown tables.

Final answer:
""".strip()

    messages = [
        {
            "role": "system",
            "content": (
                "You are TableWise, a grounded restaurant recommendation assistant. "
                "Use only the retrieved evidence. Never invent restaurant names or unsupported facts."
            ),
        },
        {
            "role": "user",
            "content": user_content,
        },
    ]

    return messages


def generate_hf_chat_answer(messages, max_new_tokens=HF_MAX_NEW_TOKENS):
    if not USE_HF_LLM_GENERATOR:
        return None

    if hasattr(hf_tokenizer, "apply_chat_template"):
        prompt_text = hf_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    else:
        prompt_text = "\n\n".join([f"{m['role'].upper()}: {m['content']}" for m in messages])
        prompt_text += "\n\nASSISTANT:"

    inputs = hf_tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        max_length=4096,
    )

    device = next(hf_model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        generated_ids = hf_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=hf_tokenizer.eos_token_id,
            eos_token_id=hf_tokenizer.eos_token_id,
        )

    input_length = inputs["input_ids"].shape[-1]
    new_tokens = generated_ids[0][input_length:]

    answer = hf_tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    ).strip()

    return answer

In [ ]:
def normalize_name_for_check(name):
    return normalize_text(name)


def find_potential_restaurant_name_lines(answer):
    candidates = []

    for m in re.finditer(r"\*\*(.*?)\*\*", answer):
        val = m.group(1).strip()
        if val:
            candidates.append(val)

    for line in answer.splitlines():
        line = line.strip()

        m = re.match(r"^\d+\.\s+\*\*(.*?)\*\*", line)
        if m:
            val = m.group(1).strip()
            if val:
                candidates.append(val)
            continue

        m = re.match(r"^\d+\.\s+(.+?)(?:\s+—|\s+-|\s*:|$)", line)
        if m:
            val = m.group(1).strip()
            val = val.strip("*").strip()
            if val:
                candidates.append(val)

    cleaned = []
    seen = set()

    for c in candidates:
        c_norm = normalize_text(c)
        if c_norm and c_norm not in seen:
            cleaned.append(c)
            seen.add(c_norm)

    return cleaned


def extract_allowed_name_mentions(answer, allowed_names):
    answer_norm = normalize_text(answer)

    mentions = []

    for name in allowed_names:
        name_norm = normalize_name_for_check(name)

        if name_norm and name_norm in answer_norm:
            mentions.append(name)

    return mentions


def check_groundedness(answer, allowed_names):
    if answer is None:
        return {
            "allowed_names": allowed_names,
            "supported_mentions": [],
            "potential_name_mentions": [],
            "unsupported_mentions": [],
            "is_grounded": False,
            "groundedness_rate": None,
        }

    allowed_norms = {normalize_name_for_check(n): n for n in allowed_names}
    mentioned_candidates = find_potential_restaurant_name_lines(answer)

    unsupported = []

    for candidate in mentioned_candidates:
        cand_norm = normalize_name_for_check(candidate)

        if cand_norm not in allowed_norms:
            is_supported = any(
                cand_norm in allowed or allowed in cand_norm
                for allowed in allowed_norms.keys()
            )

            if not is_supported:
                unsupported.append(candidate)

    supported_mentions = extract_allowed_name_mentions(answer, allowed_names)

    groundedness = {
        "allowed_names": allowed_names,
        "supported_mentions": supported_mentions,
        "potential_name_mentions": mentioned_candidates,
        "unsupported_mentions": unsupported,
        "is_grounded": len(unsupported) == 0,
        "groundedness_rate": None,
    }

    if len(mentioned_candidates) > 0:
        groundedness["groundedness_rate"] = (
            (len(mentioned_candidates) - len(unsupported)) / len(mentioned_candidates)
        )

    return groundedness

In [ ]:
def tablewise_full_pipeline(
    query,
    retrieval_top_k=100,
    baseline_top_k=20,
    final_top_k=5,
    max_recommendations=3,
    use_llm=True,
    verbose=True,
):
    total_start = time.time()

    slm_output = slm_parse_query(query)

    if slm_output["validated_json"] is None:
        parsed_query = make_empty_parsed_query()
        parser_method = "slm_failed_empty_fallback"
    else:
        parsed_query = slm_output["validated_json"]
        parser_method = "fine_tuned_slm"

    ranking_output = retrieve_rerank_and_feedback_rank(
        query=query,
        parsed_query=parsed_query,
        retrieval_top_k=retrieval_top_k,
        baseline_top_k=baseline_top_k,
        final_top_k=final_top_k,
        verbose=verbose,
    )

    feedback_results = ranking_output["feedback_results"]
    baseline_results = ranking_output["baseline_results"]

    rag_context = build_rag_context(
        feedback_results,
        top_n=final_top_k,
    )

    template_answer = generate_template_answer(
        query=query,
        parsed_query=parsed_query,
        rag_context=rag_context,
        max_recommendations=max_recommendations,
    )

    chosen_answer = template_answer
    generation_method = "template"

    llm_answer = None
    llm_groundedness = None
    llm_error = None

    if use_llm and USE_HF_LLM_GENERATOR:
        try:
            messages = build_strict_hf_prompt(
                query=query,
                parsed_query=parsed_query,
                rag_context=rag_context,
                max_recommendations=max_recommendations,
            )

            llm_answer = generate_hf_chat_answer(messages)

            llm_groundedness = check_groundedness(
                llm_answer,
                rag_context["allowed_restaurant_names"],
            )

            if llm_groundedness["is_grounded"]:
                chosen_answer = llm_answer
                generation_method = "hf_llm_grounded"
            else:
                chosen_answer = template_answer
                generation_method = "template_fallback_due_to_ungrounded_llm"

        except Exception as e:
            llm_error = repr(e)
            chosen_answer = template_answer
            generation_method = "template_fallback_due_to_llm_error"

    final_groundedness = check_groundedness(
        chosen_answer,
        rag_context["allowed_restaurant_names"],
    )

    total_elapsed = time.time() - total_start

    return {
        "query": query,
        "parser_method": parser_method,
        "slm_output": slm_output,
        "parsed_query": parsed_query,
        "ranking_output": ranking_output,
        "baseline_results": baseline_results,
        "feedback_results": feedback_results,
        "rag_context": rag_context,
        "template_answer": template_answer,
        "llm_answer": llm_answer,
        "llm_error": llm_error,
        "llm_groundedness": llm_groundedness,
        "answer": chosen_answer,
        "generation_method": generation_method,
        "final_groundedness": final_groundedness,
        "elapsed_seconds": total_elapsed,
    }

In [ ]:
demo_query = "Can you recommend me a cheap vegetarian Italian restaurant in Rome?"

out = tablewise_full_pipeline(
    query=demo_query,
    retrieval_top_k=100,
    baseline_top_k=20,
    final_top_k=5,
    max_recommendations=3,
    use_llm=True,
    verbose=True,
)

print("=" * 120)
print("USER QUERY:")
print(out["query"])

print("\nSLM RAW OUTPUT:")
print(out["slm_output"]["raw_output"])

print("\nVALIDATED PARSED QUERY:")
print(json.dumps(out["parsed_query"], indent=2, ensure_ascii=False))

print("\nBASELINE TOP 5:")
display(out["baseline_results"][
    [
        "rerank_position",
        "name",
        "city_filled",
        "price_bucket",
        "rating",
        "final_rerank_score",
        "constraint_match_reasons",
    ]
].head(5))

print("\nFEEDBACK-AWARE TOP 5:")
display(out["feedback_results"][
    [
        "feedback_rank_position",
        "name",
        "city_filled",
        "price_bucket",
        "rating",
        "final_rerank_score",
        "reward_score",
        "feedback_rerank_score",
        "constraint_match_reasons",
    ]
].head(5))

print("\nRAG CONTEXT:")
print(out["rag_context"]["context_text"])

print("\nGENERATION METHOD:", out["generation_method"])

if out["llm_error"]:
    print("\nLLM ERROR:")
    print(out["llm_error"])

print("\nFINAL ANSWER:")
print(out["answer"])

print("\nGROUNDEDNESS:")
print(json.dumps(out["final_groundedness"], indent=2, ensure_ascii=False))

print("\nElapsed seconds:", out["elapsed_seconds"])

Initial rows: 1033798
Parsed query:
{
  "city": "rome",
  "country": null,
  "price_bucket": "cheap",
  "tags": [
    "italian"
  ],
  "dietary": [
    "vegetarian friendly"
  ],
  "matched_meals": [],
  "matched_features": []
}
City filter city=rome: 1033798 -> 12603
Metadata score > 0 filter: 12603 -> 10182
After filtering: 10182
Computing semantic scores for 10,182 candidates...
USER QUERY:
Can you recommend me a cheap vegetarian Italian restaurant in Rome?

SLM RAW OUTPUT:
{"city": "rome", "country": null, "price_bucket": "cheap", "tags": ["italian"], "dietary": ["vegetarian friendly"], "matched_meals": [], "matched_features": []}

VALIDATED PARSED QUERY:
{
  "city": "rome",
  "country": null,
  "price_bucket": "cheap",
  "tags": [
    "italian"
  ],
  "dietary": [
    "vegetarian friendly"
  ],
  "matched_meals": [],
  "matched_features": []
}

BASELINE TOP 5:


,rerank_position,name,city_filled,price_bucket,rating,final_rerank_score,constraint_match_reasons
0,1,Sfiziarte - The Art of Food,Rome,cheap,5.0,0.903552,"[city=rome, diet=vegetarian friendly, price=cheap, tag=italian]"
1,2,Bacio Di Puglia,Rome,cheap,4.5,0.861916,"[city=rome, diet=vegetarian friendly, price=cheap, tag=italian]"
2,3,Pizza & Friends,Rome,cheap,5.0,0.860034,"[city=rome, diet=vegetarian friendly, price=cheap, tag=italian]"
3,4,Sto Bene Roma,Rome,cheap,4.5,0.852705,"[city=rome, diet=vegetarian friendly, price=cheap, tag=italian]"
4,5,Ristorcaffe Vecchia Roma,Rome,cheap,4.5,0.843851,"[city=rome, diet=vegetarian friendly, price=cheap, tag=italian]"



FEEDBACK-AWARE TOP 5:


,feedback_rank_position,name,city_filled,price_bucket,rating,final_rerank_score,reward_score,feedback_rerank_score,constraint_match_reasons
0,1,Sfiziarte - The Art of Food,Rome,cheap,5.0,0.903552,0.985296,0.928075,"[city=rome, diet=vegetarian friendly, price=cheap, tag=italian]"
1,2,Bacio Di Puglia,Rome,cheap,4.5,0.861916,0.965940,0.893123,"[city=rome, diet=vegetarian friendly, price=cheap, tag=italian]"
2,3,Pizza & Friends,Rome,cheap,5.0,0.860034,0.970302,0.893115,"[city=rome, diet=vegetarian friendly, price=cheap, tag=italian]"
3,4,Sto Bene Roma,Rome,cheap,4.5,0.852705,0.960186,0.884949,"[city=rome, diet=vegetarian friendly, price=cheap, tag=italian]"
4,5,Ristorcaffe Vecchia Roma,Rome,cheap,4.5,0.843851,0.953792,0.876834,"[city=rome, diet=vegetarian friendly, price=cheap, tag=italian]"



RAG CONTEXT:
[1] Sfiziarte - The Art of Food
Location: Rome, Italy
Rating: 5.0/5
Price bucket: cheap
Cuisines/tags: Cheap Eats, Italian, Mediterranean, European
Special diets: Vegetarian Friendly, Vegan Options, Gluten Free Options
Meals: Breakfast, Lunch, Dinner, Brunch
Matched constraints: city=rome, diet=vegetarian friendly, price=cheap, tag=italian
Initial rerank score: 0.9036
Feedback reward score: 0.9853

[2] Bacio Di Puglia
Location: Rome, Italy
Rating: 4.5/5
Price bucket: cheap
Cuisines/tags: Cheap Eats, Italian, Fast food, Mediterranean
Special diets: Vegetarian Friendly
Meals: Lunch, Dinner, Brunch, After-hours
Matched constraints: city=rome, diet=vegetarian friendly, price=cheap, tag=italian
Initial rerank score: 0.8619
Feedback reward score: 0.9659

[3] Pizza & Friends
Location: Rome, Italy
Rating: 5.0/5
Price bucket: cheap
Cuisines/tags: Cheap Eats, Italian, Pizza, Vegetarian Friendly
Special diets: Vegetarian Friendly, Vegan Options
Meals: Lunch, Dinner
Matched constrain

In [ ]:
demo_queries = [
    "Can you recommend me a cheap vegetarian Italian restaurant in Rome?",
    "I want an expensive fine dining place in Milan.",
    "Find me a vegetarian brunch place in Barcelona.",
    "Can you suggest tapas in Madrid?",
    "I need coffee and breakfast in Athens.",
    "Do you know any affordable seafood restaurants in Lisbon with outdoor seating?",
    "Somewhere romantic for dinner in Paris, not too expensive.",
    "Can you recommend a family-friendly restaurant in London?",
]

batch_outputs = {}
batch_rows = []

for q in demo_queries:
    print("=" * 120)
    print("Running:", q)

    out = tablewise_full_pipeline(
        query=q,
        retrieval_top_k=100,
        baseline_top_k=20,
        final_top_k=5,
        max_recommendations=3,
        use_llm=True,
        verbose=False,
    )

    batch_outputs[q] = out

    feedback_results = out["feedback_results"]

    top_names = []
    if len(feedback_results) > 0:
        top_names = feedback_results["name"].head(3).tolist()

    batch_rows.append({
        "query": q,
        "parser_method": out["parser_method"],
        "parsed_city": out["parsed_query"].get("city"),
        "parsed_price_bucket": out["parsed_query"].get("price_bucket"),
        "parsed_tags": out["parsed_query"].get("tags"),
        "parsed_dietary": out["parsed_query"].get("dietary"),
        "num_final_results": len(feedback_results),
        "top_3_restaurants": top_names,
        "generation_method": out["generation_method"],
        "is_grounded": out["final_groundedness"]["is_grounded"],
        "elapsed_seconds": out["elapsed_seconds"],
        "answer_preview": out["answer"][:350],
    })

batch_df = pd.DataFrame(batch_rows)
display(batch_df)

Running: Can you recommend me a cheap vegetarian Italian restaurant in Rome?
Running: I want an expensive fine dining place in Milan.
Running: Find me a vegetarian brunch place in Barcelona.
Running: Can you suggest tapas in Madrid?
Running: I need coffee and breakfast in Athens.
Running: Do you know any affordable seafood restaurants in Lisbon with outdoor seating?
Running: Somewhere romantic for dinner in Paris, not too expensive.
Running: Can you recommend a family-friendly restaurant in London?


,query,parser_method,parsed_city,parsed_price_bucket,parsed_tags,parsed_dietary,num_final_results,top_3_restaurants,generation_method,is_grounded,elapsed_seconds,answer_preview
0,Can you recommend me a cheap vegetarian Italian restaurant in Rome?,fine_tuned_slm,rome,cheap,[italian],[vegetarian friendly],5,"[Sfiziarte - The Art of Food, Bacio Di Puglia, Pizza & Friends]",hf_llm_grounded,True,10.649534,"Sfiziarte - The Art of Food, Bacio Di Puglia, and Pizza & Friends are all highly recommended vegetarian Italian restaurants in Rome that fit your criteria of being cheap, with options for breakfast, lunch, dinner, and brunch."
1,I want an expensive fine dining place in Milan.,fine_tuned_slm,milan,expensive,[fine dining],[],5,"[Seta, Ba Restaurant, Langosteria Café]",hf_llm_grounded,True,20.639394,"Based on the provided information and constraints, here are three highly recommended expensive fine dining restaurants in Milan:\n\n1. **Seta**\n - Location: Milan, Italy\n - Rating: 4.5/5\n - Price bucket: expensive\n - Cuisines/tags: Fine Dining, Italian, Seafood, Mediterranean\n - Special diets: Vegetarian Friendly, Vegan Options, Gluten Free"
2,Find me a vegetarian brunch place in Barcelona.,fine_tuned_slm,barcelona,None,[],[vegetarian friendly],5,"[Eixampeling Brunch Café & Bar, Ugot Bruncherie, Bo Kaap]",hf_llm_grounded,True,11.003181,Eixampeling Brunch Café & Bar\nUgot Bruncherie\nBo Kaap
3,Can you suggest tapas in Madrid?,fine_tuned_slm,madrid,None,[tapas],[],5,"[Tapa Tapa Arenal, Tacos & Tapas, Tinto y Tapas San Pedro]",hf_llm_grounded,True,15.856434,"Based on the available information, here are three options for tapas in Madrid:\n\n1. **Tapa Tapa Arenal**\n - Location: Madrid, Spain\n - Rating: 4.5/5\n - Price bucket: expensive\n - Cuisines/tags: Mid-range, Spanish, Vegetarian Friendly, Gluten Free Options\n\n2. **Tacos & Tapas**\n - Location: Madrid, Spain\n - Rating: 4.5/5\n - Price bucket"
4,I need coffee and breakfast in Athens.,fine_tuned_slm,athens,None,[coffee],[],5,"[Coffee Joint, Victory Cafe, Kekkos Traditional Cafe and Pastry]",hf_llm_grounded,True,21.029202,"Based on your preferences for coffee and breakfast in Athens, here are three recommended restaurants:\n\n1. **Coffee Joint**\n - Location: Athens, Greece\n - Rating: 5.0/5\n - Price bucket: cheap\n - Cuisines/tags: Cheap Eats, Cafe, Mediterranean, Greek\n - Special diets: Vegetarian Friendly, Vegan Options\n - Meals: Breakfast, Lunch, Brunch\n"
5,Do you know any affordable seafood restaurants in Lisbon with outdoor seating?,fine_tuned_slm,lisbon,cheap,[seafood],[],5,"[Maca Verde, Time Out Market Lisboa, Cais ao Mar - Cervejaria Marisqueira]",hf_llm_grounded,True,9.699989,"Maca Verde, Time Out Market Lisboa, and Cais ao Mar - Cervejaria Marisqueira are affordable seafood restaurants in Lisbon with outdoor seating."
6,"Somewhere romantic for dinner in Paris, not too expensive.",fine_tuned_slm,paris,mid,[],[],5,"[La Grande Ourse, La P'tite Table, La Lingerie]",hf_llm_grounded,True,20.429006,"Based on the user's query for a romantic dinner in Paris that is not too expensive, here are three recommended restaurants:\n\n1. **La Grande Ourse**\n - Location: Paris, France\n - Rating: 4.5/5\n - Price bucket: mid\n - Cuisines/tags: Mid-range, French, European\n - Meals: Lunch, Dinner\n - Features: Reservations, Outdoor Seating, Seating, Wh"
7,Can you recommend a family-friendly restaurant in London?,fine_tuned_slm,london,None,[],[],0,[],hf_llm_grounded,True,7.967312,"Sorry, but no family-friendly restaurants were found in the dataset for London."


In [ ]:
for q, out in batch_outputs.items():
    print("=" * 120)
    print("USER QUERY:")
    print(q)

    print("\nPARSED QUERY:")
    print(json.dumps(out["parsed_query"], indent=2, ensure_ascii=False))

    print("\nTOP RESTAURANTS AFTER FEEDBACK RERANKING:")
    if len(out["feedback_results"]) > 0:
        display(out["feedback_results"][
            [
                "feedback_rank_position",
                "name",
                "city_filled",
                "price_bucket",
                "rating",
                "reward_score",
                "feedback_rerank_score",
            ]
        ].head(5))
    else:
        print("No results.")

    print("\nFINAL ANSWER:")
    print(out["answer"])

    print("\nGENERATION METHOD:", out["generation_method"])
    print("GROUNDED:", out["final_groundedness"]["is_grounded"])
    print()

USER QUERY:
Can you recommend me a cheap vegetarian Italian restaurant in Rome?

PARSED QUERY:
{
  "city": "rome",
  "country": null,
  "price_bucket": "cheap",
  "tags": [
    "italian"
  ],
  "dietary": [
    "vegetarian friendly"
  ],
  "matched_meals": [],
  "matched_features": []
}

TOP RESTAURANTS AFTER FEEDBACK RERANKING:


,feedback_rank_position,name,city_filled,price_bucket,rating,reward_score,feedback_rerank_score
0,1,Sfiziarte - The Art of Food,Rome,cheap,5.0,0.985296,0.928075
1,2,Bacio Di Puglia,Rome,cheap,4.5,0.965940,0.893123
2,3,Pizza & Friends,Rome,cheap,5.0,0.970302,0.893115
3,4,Sto Bene Roma,Rome,cheap,4.5,0.960186,0.884949
4,5,Ristorcaffe Vecchia Roma,Rome,cheap,4.5,0.953792,0.876834



FINAL ANSWER:
Sfiziarte - The Art of Food, Bacio Di Puglia, and Pizza & Friends are all highly recommended vegetarian Italian restaurants in Rome that fit your criteria of being cheap, with options for breakfast, lunch, dinner, and brunch.

GENERATION METHOD: hf_llm_grounded
GROUNDED: True

USER QUERY:
I want an expensive fine dining place in Milan.

PARSED QUERY:
{
  "city": "milan",
  "country": null,
  "price_bucket": "expensive",
  "tags": [
    "fine dining"
  ],
  "dietary": [],
  "matched_meals": [],
  "matched_features": []
}

TOP RESTAURANTS AFTER FEEDBACK RERANKING:


,feedback_rank_position,name,city_filled,price_bucket,rating,reward_score,feedback_rerank_score
0,1,Seta,Milan,expensive,4.5,0.978000,0.907318
1,2,Ba Restaurant,Milan,expensive,4.5,0.965225,0.881539
2,3,Langosteria Café,Milan,expensive,4.5,0.965414,0.880974
3,4,Frades Porto Cervo - Milano,Milan,expensive,5.0,0.965645,0.875466
4,5,Il Salotto Del Pesce Crudo & Cocktail,Milan,expensive,5.0,0.961818,0.868872



FINAL ANSWER:
Based on the provided information and constraints, here are three highly recommended expensive fine dining restaurants in Milan:

1. **Seta**
   - Location: Milan, Italy
   - Rating: 4.5/5
   - Price bucket: expensive
   - Cuisines/tags: Fine Dining, Italian, Seafood, Mediterranean
   - Special diets: Vegetarian Friendly, Vegan Options, Gluten Free Options
   - Meals: Lunch, Dinner
   - Features: Outdoor Seating, Reservations, Private Dining, Seating, Parking Available, Validated Parking, Valet Parking, Wheelchair Accessible, Serves Alcohol, Full Bar, Accepts American Express, Accepts Mastercard, Accepts Visa, Free Wifi, Accepts Credit Cards, Table Service

2. **Ba Restaurant**
   - Location: Milan, Italy
   - Rating: 4.5/5
   - Price bucket: expensive
   - Cuisines/tags: Fine Dining, Chinese, Seafood, Asian
   - Special diets: Vegetarian Friendly, Vegan Options, Gluten Free Options
   - Meals: Lunch, Dinner, After-hours
   - Features: Reservations, Seating, Wheelchair A

...


FINAL ANSWER:
Based on your preferences for coffee and breakfast in Athens, here are three recommended restaurants:

1. **Coffee Joint**
   - Location: Athens, Greece
   - Rating: 5.0/5
   - Price bucket: cheap
   - Cuisines/tags: Cheap Eats, Cafe, Mediterranean, Greek
   - Special diets: Vegetarian Friendly, Vegan Options
   - Meals: Breakfast, Lunch, Brunch
   - Features: Takeout, Seating, Wheelchair Accessible, Table Service

2. **Victory Cafe**
   - Location: Athens, Greece
   - Rating: 5.0/5
   - Price bucket: mid
   - Cuisines/tags: Cheap Eats, Cafe, European, Healthy
   - Special diets: Vegetarian Friendly, Vegan Options, Gluten Free Options
   - Meals: Breakfast, Lunch, Dinner, Brunch
   - Features: Takeout, Seating, Wheelchair Accessible, Serves Alcohol, Full Bar, Free Wifi, Table Service

3. **Kekkos Traditional Cafe and Pastry**
   - Location: Athens, Greece
   - Rating: 5.0/5
   - Price bucket: cheap
   - Cuisines/tags: Cheap Eats, Bakeries, Cafe, Mediterranean
   - Specia

,feedback_rank_position,name,city_filled,price_bucket,rating,reward_score,feedback_rerank_score
0,1,Maca Verde,Lisbon,cheap,4.5,0.906328,0.825195
1,2,Time Out Market Lisboa,Lisbon,mid,4.5,0.817581,0.768241
2,3,Cais ao Mar - Cervejaria Marisqueira,Lisbon,expensive,4.5,0.808089,0.763567
3,4,A Cevicheria,Lisbon,mid,4.5,0.787069,0.753346
4,5,Restaurante Ortónimo,Lisbon,expensive,4.5,0.786521,0.752129



FINAL ANSWER:
Maca Verde, Time Out Market Lisboa, and Cais ao Mar - Cervejaria Marisqueira are affordable seafood restaurants in Lisbon with outdoor seating.

GENERATION METHOD: hf_llm_grounded
GROUNDED: True

USER QUERY:
Somewhere romantic for dinner in Paris, not too expensive.

PARSED QUERY:
{
  "city": "paris",
  "country": null,
  "price_bucket": "mid",
  "tags": [],
  "dietary": [],
  "matched_meals": [
    "dinner"
  ],
  "matched_features": [
    "romantic"
  ]
}

TOP RESTAURANTS AFTER FEEDBACK RERANKING:


,feedback_rank_position,name,city_filled,price_bucket,rating,reward_score,feedback_rerank_score
0,1,La Grande Ourse,Paris,mid,4.5,0.916185,0.819189
1,2,La P'tite Table,Paris,mid,4.5,0.857964,0.777138
2,3,La Lingerie,Paris,mid,4.5,0.853483,0.776667
3,4,L'Amour Vache,Paris,mid,4.0,0.827498,0.768157
4,5,Royal Madeleine,Paris,mid,4.0,0.812631,0.757020



FINAL ANSWER:
Based on the user's query for a romantic dinner in Paris that is not too expensive, here are three recommended restaurants:

1. **La Grande Ourse**
   - Location: Paris, France
   - Rating: 4.5/5
   - Price bucket: mid
   - Cuisines/tags: Mid-range, French, European
   - Meals: Lunch, Dinner
   - Features: Reservations, Outdoor Seating, Seating, Wheelchair Accessible, Serves Alcohol, Accepts Credit Cards, Table Service

2. **La P'tite Table**
   - Location: Paris, France
   - Rating: 4.5/5
   - Price bucket: mid
   - Cuisines/tags: Mid-range, French, Cafe
   - Meals: Lunch, Dinner
   - Features: Reservations, Seating, Table Service

3. **L'Amour Vache**
   - Location: Paris, France
   - Rating: 4.0/5
   - Price bucket: mid
   - Cuisines/tags: Mid-range, French, Bar, European
   - Special diets: Vegetarian Friendly
   - Meals: Breakfast, Lunch, Dinner, Brunch

These options should provide a romantic dining experience without breaking the bank in Paris.

GENERATION METHOD:

In [ ]:
final_metrics = {
    "num_demo_queries": len(batch_df),
    "grounded_answer_rate": float(batch_df["is_grounded"].mean()),
    "mean_latency_seconds": float(batch_df["elapsed_seconds"].mean()),
    "num_no_result_queries": int((batch_df["num_final_results"] == 0).sum()),
    "llm_generation_rate": float(batch_df["generation_method"].str.contains("hf_llm").mean()),
    "template_fallback_rate": float(batch_df["generation_method"].str.contains("template").mean()),
}

final_metrics_df = pd.DataFrame([final_metrics])

display(final_metrics_df)

,num_demo_queries,grounded_answer_rate,mean_latency_seconds,num_no_result_queries,llm_generation_rate,template_fallback_rate
0,8,1.0,14.659256,1,1.0,0.0


In [ ]:
BATCH_RESULTS_PATH = OUTPUT_DIR / "final_pipeline_batch_results.csv"
FINAL_METRICS_PATH = OUTPUT_DIR / "final_pipeline_metrics.csv"
FINAL_ANSWERS_PATH = OUTPUT_DIR / "final_pipeline_answers.json"

batch_df.to_csv(BATCH_RESULTS_PATH, index=False)
final_metrics_df.to_csv(FINAL_METRICS_PATH, index=False)

answers_payload = []

for q, out in batch_outputs.items():
    answers_payload.append({
        "query": q,
        "parser_method": out["parser_method"],
        "slm_raw_output": out["slm_output"]["raw_output"],
        "parsed_query": out["parsed_query"],
        "top_restaurants": out["feedback_results"]["name"].head(5).tolist() if len(out["feedback_results"]) > 0 else [],
        "generation_method": out["generation_method"],
        "answer": out["answer"],
        "groundedness": out["final_groundedness"],
        "elapsed_seconds": out["elapsed_seconds"],
    })

with open(FINAL_ANSWERS_PATH, "w", encoding="utf-8") as f:
    json.dump(answers_payload, f, indent=2, ensure_ascii=False)

print("Saved:")
print("-", BATCH_RESULTS_PATH)
print("-", FINAL_METRICS_PATH)
print("-", FINAL_ANSWERS_PATH)

Saved:
- /content/drive/MyDrive/tablewise/artifacts_new/final_pipeline/final_pipeline_batch_results.csv
- /content/drive/MyDrive/tablewise/artifacts_new/final_pipeline/final_pipeline_metrics.csv
- /content/drive/MyDrive/tablewise/artifacts_new/final_pipeline/final_pipeline_answers.json
